In [68]:
from os import chmod

import pandas as pd
import numpy as np
import networkx as nx
from func.UsefulFunctions import *

In [69]:
#Start with the data for artists

artistDF = pd.read_parquet("Data/Network_artistsWikidata.parquet")


We will be linking artists to their influences, record labels, genres.

In [70]:
artistDF.head()

,artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
0,Q3176190,Q639669,Q6581072,musician,female,Q30,NaN,NaN,NaN,Q43343,https://en.wikipedia.org/wiki/Linda_Gail_Lewis,a2fe484f-4f35-4e0e-a954-2f6d5843b009,Q7544232,Smash Records,United States,folk music,Linda Gail Lewis
1,Q8065552,Q183945,Q6581097,record producer,male,Q145,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Zalon,19c43212-18cf-4f02-baff-b794de3cdf02,NaN,NaN,United Kingdom,NaN,Zalon
2,Q191819,Q806349,Q6581097,bandleader,male,Q174193,NaN,NaN,NaN,Q9730,https://en.wikipedia.org/wiki/George_Martin,26fa8b67-6c7f-406c-ad64-a1d070092df2,Q21077,Warner Music Group,United Kingdom of Great Britain and Ireland,classical music,George Martin
3,Q21934026,Q183945,Q6581097,record producer,male,NaN,NaN,NaN,NaN,Q37073,https://en.wikipedia.org/wiki/Andrew_Scheps,ee11737c-d50c-4b04-ae5f-56e1f6373116,NaN,NaN,NaN,pop music,Andrew Scheps
4,Q1005957,Q55960555,Q6581097,recording artist,male,Q28,NaN,NaN,NaN,NaN,NaN,6054baf8-3a47-4c3a-8c51-eb976f69ecfc,NaN,NaN,Hungary,NaN,László Földes


In [71]:
# artistDF["Node_Name"] = artistDF["artist"] + "-" +artistDF["artistLabel"]
# artistDF["Genre_Name"] = artistDF["genre"] + "-" +artistDF["genreLabel"]
# artistDF["recordLabel_Name"] = artistDF["recordLabel"] + "-" +artistDF["recordLabelLabel"]
# artistDF["Influence_Name"] = artistDF["influencedBY"] + "-" +artistDF["influencedBYLabel"]


In [72]:
graphInfluence = addToNetwork4(artistDF, "influencedBY", "artist", "has influenced", "Musical Artist", "Musical artist","influencedBYLabel","artistLabel")


In [73]:
graphRecordLabel = addToNetwork4(artistDF, "recordLabel", "artist", "has/had artist","Record Label", "Musical Artist", "recordLabelLabel", "artistLabel")

In [74]:
graphGenre = addToNetwork4(artistDF, "genre", "artist", "has been participated in by", "Genre", "Musical Artist", "genreLabel", "artistLabel")

In [75]:
#compose all the graphs
firstGraph = nx.compose(graphInfluence, graphRecordLabel)
artistGraph = nx.compose(firstGraph, graphGenre)


In [76]:
#See the nodes in artistGraph
artistGraph.nodes["Q122971758"]

{'EntityType': 'Musical Artist', 'Label': 'Ikkimel'}

In [77]:
nx.write_graphml(artistGraph, "./graphs/artistGraph.graphml")

In [78]:
artistGraph.to_undirected().edges("Q5084390")

EdgeDataView([('Q5084390', 'Q42455'), ('Q5084390', 'Q11975'), ('Q5084390', 'Q21077'), ('Q5084390', 'Q202440'), ('Q5084390', 'Q1998195'), ('Q5084390', 'Q2690060'), ('Q5084390', 'Q37073'), ('Q5084390', 'Q187760'), ('Q5084390', 'Q842324'), ('Q5084390', 'Q188450'), ('Q5084390', 'Q487914'), ('Q5084390', 'Q211756'), ('Q5084390', 'Q1298934'), ('Q5084390', 'Q104695865'), ('Q5084390', 'Q9162904')])

In [79]:
ikkimelEgo = nx.ego_graph(artistGraph, "Q122971758", radius=3, center=True, undirected = True).copy()
print(ikkimelEgo.number_of_nodes())

39


In [80]:
sufjanEgo = nx.ego_graph(artistGraph, "Q319502", radius=3, center=True, undirected = True).copy()
print(sufjanEgo.number_of_nodes())
nx.write_graphml(sufjanEgo, "./graphs/sufjanEgo.graphml")

15549


In [81]:
#Subgraph of bjork, fka twigs, electronic music, hip-hop, arca
twigsEgoGraph = nx.ego_graph(artistGraph, "Q15291033", radius=3, center=True, undirected = True).copy()
#electronicEgoGraph = nx.ego_graph(artistGraph, "Q9778", radius=2, center=True,undirected = True).copy()
#subgraph = nx.compose(electronicEgoGraph, bjorkEgoGraph)
print(len(twigsEgoGraph.nodes()))
nx.write_graphml(twigsEgoGraph, "./graphs/twigsEgoGraph.graphml")
nx.write_graphml(ikkimelEgo, "./graphs/ikkimelEgo.graphml")

591


Make a triphop ego network

In [82]:
#Create a triphop ego network for artists of 1 away
trip_hop_WikidataEgo = nx.ego_graph(artistGraph, "Q205560", radius=1, center=True, undirected = True).copy()
nx.write_graphml(trip_hop_WikidataEgo, "./graphs/trip_hop_WikidataEgo.graphml")

In [83]:
#Create a trip hop ego network for from the data from mb
mbPersonData = pl.read_parquet("./Data/Network_personMBData.parquet")
#explode genre list to get only trip hop
mbPersonData = mbPersonData.explode("genreList", empty_as_null=True)
mbPersonData = mbPersonData.unique()
mbPersonData.head()

id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
str,str,str,str,str,list[struct[4]],list[str],list[str],struct[3],str
"""3f694a16-e2ee-4705-9d89-c8f9f6…","""Benjamin Bernstein""","""Person""","""Male""",null,[],[],[],"{null,null,false}",null
"""d9a0e6a0-bef7-449b-a39f-259ee4…","""PRGMAT""","""Person""","""Male""","""LV""","[{""d8048b43-9edb-4e1a-aa88-bb91fa847d4a"",1,"""",""boom bap""}, {""18d52a87-bbbf-42f0-8338-189c8c314523"",1,"""",""chillwave""}, … {""d92aaf30-5941-4a65-b741-d9d48c80772a"",1,"""",""lo-fi hip hop""}]",[],[],"{null,null,false}","""lo-fi"""
"""25d77850-61db-4345-8ce2-496d14…","""Paul Cleary""","""Person""",null,null,"[{""911c7bbb-172d-4df8-9478-dbff4296e791"",1,"""",""pop""}]",[],[],"{null,null,false}","""pop"""
"""dddfca35-ca07-44d9-8732-00a73f…","""Secret School""","""Person""","""Male""","""US""",[],[],[],"{null,null,false}",null
"""23038456-fae2-4fae-99de-9e6431…","""片山義美""","""Person""","""Female""","""JP""",[],[],[],"{null,""1987-03-17"",false}",null


In [84]:
#On musicbrainz the genre tripHop is called tripHop
tripHopPeople = mbPersonData.filter(
    pl.col("genreList") == "trip hop"
).select(
    pl.col("id"),
    pl.col("name"),
    pl.col("genreList").alias("genreLabel")
).unique()

tripHopPeople.head()

id,name,genreLabel
str,str,str
"""71ebee16-5d83-40eb-a699-ff063a…","""Al Lover""","""trip hop"""
"""bc2706d9-533c-4f98-9dd2-6338d2…","""King Jase""","""trip hop"""
"""bff0b8e6-5c70-43a5-b01e-0d825e…","""Andrea Parker""","""trip hop"""
"""0c445197-bab3-442a-b1d2-32963f…","""Phutureprimitive""","""trip hop"""
"""640485e3-2b3a-4495-b35a-6816e1…","""Forever""","""trip hop"""


In [85]:
#Lets attempt to find them on wikidata

tripHopPeoplePandas = tripHopPeople.to_pandas()
tripHopPeoplePandas = tripHopPeoplePandas.merge(artistDF[["artist", "musicBrainzID"]], how="left", left_on="id", right_on="musicBrainzID")
tripHopPeoplePandas.drop_duplicates(inplace=True)
tripHopPeoplePandas.head()

,id,name,genreLabel,artist,musicBrainzID
0,71ebee16-5d83-40eb-a699-ff063a01d961,Al Lover,trip hop,NaN,NaN
1,bc2706d9-533c-4f98-9dd2-6338d2d88702,King Jase,trip hop,Q115259076,bc2706d9-533c-4f98-9dd2-6338d2d88702
2,bff0b8e6-5c70-43a5-b01e-0d825eb67821,Andrea Parker,trip hop,Q494650,bff0b8e6-5c70-43a5-b01e-0d825eb67821
12,0c445197-bab3-442a-b1d2-32963f1b39a0,Phutureprimitive,trip hop,NaN,NaN
13,640485e3-2b3a-4495-b35a-6816e11995ae,Forever,trip hop,NaN,NaN


In [86]:
#Make the musicBrainz ego network for tripHOP

#add all the q-number nodes then add all
artistsWithQNumbers= tripHopPeoplePandas[tripHopPeoplePandas.artist.notna()][["artist","name"]].drop_duplicates()
listOfQnumberTuples = artistsWithQNumbers.itertuples(index=False, name=None)
listOfQnumberTuples = [(k, {"Label": v, "EntityType": "Musical Artist"}) for k, v in listOfQnumberTuples]
#then add all the nodes of individuals without q-numbers
artistsWithOnlyID= tripHopPeoplePandas[tripHopPeoplePandas.artist.isna()][["id","name"]].drop_duplicates()
listOfIdNumberTuples = artistsWithOnlyID.itertuples(index=False, name=None)
listOfIdNumberTuples = [(k, {"Label": v, "EntityType": "Musical Artist"}) for k, v in listOfIdNumberTuples]

#connect them all to triphop

mbTripHopEgo = nx.DiGraph()
#add an vertex for triphop
mbTripHopEgo.add_node("Q205560", Label = "trip hop", EntityType = "Genre")
mbTripHopEgo.add_nodes_from(listOfIdNumberTuples)
mbTripHopEgo.add_nodes_from(listOfQnumberTuples)

for x in mbTripHopEgo.nodes:
    #add edges to triphop
    mbTripHopEgo.add_edge("Q205560", x, Type="has been participated in by (MB)")


#save the graph
nx.write_graphml(mbTripHopEgo, "./graphs/mbTripHopEgo.graphml")




In [87]:
mergedTripHop_Graph = nx.compose(mbTripHopEgo, trip_hop_WikidataEgo)
nx.write_graphml(mergedTripHop_Graph, "./graphs/mergedTripHopEgo.graphml")